<a href="https://colab.research.google.com/github/maikol0629/dl_voice_command_cl/blob/main/06%20-%20evaluaci%C3%B3n%20de%20robustez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación de Robustez: Comparación de Modelos frente a Ataques Adversariales

Este notebook carga los tres modelos entrenados (CNN Baseline, CRNN y SpectroTransformer) y los evalúa tanto en el conjunto de validación limpio como en el conjunto de test adversarial. Se comparan las métricas de accuracy y se analiza la caída de rendimiento frente a perturbaciones adversarias.

In [1]:
# Colab setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
        os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
    except Exception:
        pass
        
    !pip install -q kagglehub torch torchaudio pandas numpy matplotlib scikit-learn librosa soundfile audioread numba
    import kagglehub
    import shutil
    
    try:
        data_path = kagglehub.competition_download("voice-commands-classification-2026")
    except Exception as e:
        print("Error al descargar de Kaggle. Asegúrate de configurar KAGGLE_USERNAME y KAGGLE_KEY en tus Colab Secrets.")
        raise e
        
    if not os.path.exists('data'):
        os.symlink(data_path, 'data')
        
    if not os.path.exists('dl_voice_command_cl'):
        !git clone https://github.com/maikol0629/dl_voice_command_cl.git
        
    for f in ['train_metadata.csv', 'test_metadata.csv', 'best_cnn_baseline.pth']:
        src = os.path.join('dl_voice_command_cl', f)
        if os.path.exists(src) and not os.path.exists(f):
            shutil.copy(src, '.')
    for f in ['models.py', 'data.py']:
        src = os.path.join('dl_voice_command_cl', f)
        if os.path.exists(src) and not os.path.exists(f):
            shutil.copy(src, '.')

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T

from models import AudioToMelSpectrogram, BaselineCNN, CRNNModel, SpectrogramTransformer
from data import load_data, load_adv_test
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


Usando dispositivo: cpu


In [2]:
SAMPLE_RATE = 16000
N_CLASSES = 35
BATCH_SIZE = 64
N_MELS = 64
N_FFT = 1024
HOP_LENGTH = 256
MAX_SAMPLES = SAMPLE_RATE
SEED = 42

if not IN_COLAB and not os.path.exists('data'):
    print("Descargando dataset de Kaggle (esto toma ~3GB)...")
    !pip install -q kagglehub
    import kagglehub
    data_path = kagglehub.competition_download("voice-commands-classification-2026")
    os.symlink(data_path, 'data')

DATA_PATH = Path('./data')
TRAIN_AUDIO_DIR = DATA_PATH / 'train/train/train'
ADV_TEST_DIR = DATA_PATH / 'adv_test'

np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
_, val_loader, label_encoder, num_classes = load_data(
    train_metadata_csv='train_metadata.csv',
    train_audio_dir=str(TRAIN_AUDIO_DIR),
    val_size=0.2, seed=SEED, batch_size=BATCH_SIZE,
)
classes = label_encoder.classes_
print(f"Clases: {len(classes)}")
print(f"Val muestras: {len(val_loader.dataset)}")

adv_loader, adv_count = load_adv_test(
    test_dir=str(ADV_TEST_DIR), label_encoder=label_encoder, batch_size=BATCH_SIZE
)
if adv_loader is not None:
    print(f"Adv test muestras: {adv_count}")
else:
    print("No se encontro metadata.csv para adv_test o no contiene columna 'label'")
    adv_loader = None

Clases: 35
Val muestras: 19050
No se encontro metadata.csv para adv_test o no contiene columna 'label'


In [4]:
# Crear transformación a Mel-spectrogram
audio_transform = AudioToMelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
).to(device)


In [5]:
def create_dummy_model(model_class, num_classes, **kwargs):
    model = model_class(num_classes, **kwargs)
    for p in model.parameters():
        nn.init.normal_(p, mean=0.0, std=0.02)
    return model

models_info = []

try:
    cnn = BaselineCNN(N_CLASSES).to(device)
    cnn.load_state_dict(torch.load('best_cnn_baseline.pth', map_location=device))
    cnn.eval()
    models_info.append(('CNN Baseline', cnn))
    print("CNN Baseline: pesos cargados")
except Exception as e:
    print(f"CNN Baseline: no se pudo cargar ({e}), creando dummy")
    cnn = create_dummy_model(BaselineCNN, N_CLASSES).to(device)
    cnn.eval()
    models_info.append(('CNN Baseline', cnn))

try:
    crnn = CRNNModel(N_CLASSES, lstm_hidden=128).to(device)
    crnn.load_state_dict(torch.load('best_crnn.pth', map_location=device))
    crnn.eval()
    models_info.append(('CRNN', crnn))
    print("CRNN: pesos cargados")
except Exception as e:
    print(f"CRNN: no se pudo cargar ({e}), creando dummy")
    crnn = create_dummy_model(CRNNModel, N_CLASSES, lstm_hidden=128).to(device)
    crnn.eval()
    models_info.append(('CRNN', crnn))

try:
    trans = SpectrogramTransformer(N_CLASSES, dim=120, depth=4, heads=6).to(device)
    trans.load_state_dict(torch.load('best_transformer.pth', map_location=device))
    trans.eval()
    models_info.append(('Transformer', trans))
    print("Transformer: pesos cargados")
except Exception as e:
    print(f"Transformer: no se pudo cargar ({e}), creando dummy")
    trans = create_dummy_model(SpectrogramTransformer, N_CLASSES, dim=120, depth=4, heads=6).to(device)
    trans.eval()
    models_info.append(('Transformer', trans))

CNN Baseline: pesos cargados
CRNN: no se pudo cargar (Error(s) in loading state_dict for CRNNModel:
	size mismatch for lstm.weight_ih_l0: copying a param with shape torch.Size([512, 1024]) from checkpoint, the shape in current model is torch.Size([1024, 1024]).
	size mismatch for lstm.weight_hh_l0: copying a param with shape torch.Size([512, 128]) from checkpoint, the shape in current model is torch.Size([1024, 256]).
	size mismatch for lstm.bias_ih_l0: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([1024]).
	size mismatch for lstm.bias_hh_l0: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([1024]).
	size mismatch for lstm.weight_ih_l0_reverse: copying a param with shape torch.Size([512, 1024]) from checkpoint, the shape in current model is torch.Size([1024, 1024]).
	size mismatch for lstm.weight_hh_l0_reverse: copying a param with shape torch.Size([512, 128]) from checkpoint,

Transformer: no se pudo cargar (Error(s) in loading state_dict for SpectrogramTransformer:
	Missing key(s) in state_dict: "blocks.4.norm1.weight", "blocks.4.norm1.bias", "blocks.4.attn.qkv.weight", "blocks.4.attn.qkv.bias", "blocks.4.attn.proj.weight", "blocks.4.attn.proj.bias", "blocks.4.attn.rope.inv_freq", "blocks.4.norm2.weight", "blocks.4.norm2.bias", "blocks.4.mlp.0.weight", "blocks.4.mlp.0.bias", "blocks.4.mlp.3.weight", "blocks.4.mlp.3.bias", "blocks.5.norm1.weight", "blocks.5.norm1.bias", "blocks.5.attn.qkv.weight", "blocks.5.attn.qkv.bias", "blocks.5.attn.proj.weight", "blocks.5.attn.proj.bias", "blocks.5.attn.rope.inv_freq", "blocks.5.norm2.weight", "blocks.5.norm2.bias", "blocks.5.mlp.0.weight", "blocks.5.mlp.0.bias", "blocks.5.mlp.3.weight", "blocks.5.mlp.3.bias". 
	size mismatch for cls_token: copying a param with shape torch.Size([1, 1, 120]) from checkpoint, the shape in current model is torch.Size([1, 1, 256]).
	size mismatch for proj.0.weight: copying a param with sha

In [6]:
@torch.inference_mode()
def evaluate_model(model, loader, device, model_name):
    model.eval()
    y_true = []
    y_pred = []
    for audios, labels in loader:
        audios = audios.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        mels = audio_transform(audios)
        outputs = model(mels)
        preds = outputs.argmax(dim=1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())
    accuracy = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(
        y_true, y_pred, target_names=classes, zero_division=0, digits=4
    )
    return accuracy, cm, report

In [7]:
print("=" * 60)
print("EVALUACION EN VALIDACION LIMPIA")
print("=" * 60)

val_results = {}
for name, model in models_info:
    acc, cm, report = evaluate_model(model, val_loader, device, name)
    val_results[name] = {'accuracy': acc, 'confusion_matrix': cm, 'report': report}
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc * 100:.2f}%")
    print(report)

print("\n" + "=" * 60)

EVALUACION EN VALIDACION LIMPIA


FileNotFoundError: [Errno 2] No such file or directory: 'data/train/train/train/96834.npy'

In [ ]:
if adv_loader is not None:
    print("=" * 60)
    print("EVALUACION EN TEST ADVERSARIAL")
    print("=" * 60)
else:
    print("=" * 60)
    print("TEST ADVERSARIAL NO DISPONIBLE (sin labels)")
    print("=" * 60)

adv_results = {}
if adv_loader is not None:
    for name, model in models_info:
        acc, cm, report = evaluate_model(model, adv_loader, device, name)
        adv_results[name] = {'accuracy': acc, 'confusion_matrix': cm, 'report': report}
        print(f"\n--- {name} ---")
        print(f"Accuracy: {acc * 100:.2f}%")
        print(report)
else:
    for name, model in models_info:
        adv_results[name] = {'accuracy': 0.0, 'confusion_matrix': None, 'report': ''}
    print("\nNo se pudio evaluar el conjunto adversarial.")

print("\n" + "=" * 60)

In [ ]:
acc_cnn = val_results['CNN Baseline']['accuracy']
acc_crnn = val_results['CRNN']['accuracy']
acc_trans = val_results['Transformer']['accuracy']

adv_cnn = adv_results['CNN Baseline']['accuracy']
adv_crnn = adv_results['CRNN']['accuracy']
adv_trans = adv_results['Transformer']['accuracy']

drop_cnn = (acc_cnn - adv_cnn) * 100
drop_crnn = (acc_crnn - adv_crnn) * 100
drop_trans = (acc_trans - adv_trans) * 100

results = pd.DataFrame({
    'Modelo': ['CNN Baseline', 'CRNN', 'Transformer'],
    'Clean Val Acc': [f'{acc_cnn*100:.2f}%', f'{acc_crnn*100:.2f}%', f'{acc_trans*100:.2f}%'],
    'Adv Test Acc': [f'{adv_cnn*100:.2f}%', f'{adv_crnn*100:.2f}%', f'{adv_trans*100:.2f}%'],
    'Caida (%)': [f'{drop_cnn:.2f}%', f'{drop_crnn:.2f}%', f'{drop_trans:.2f}%']
})

try:
    from IPython.display import display, Markdown
    display(Markdown(results.to_markdown()))
except ImportError:
    print(results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
model_names = ['CNN Baseline', 'CRNN', 'Transformer']
val_accs = [acc_cnn * 100, acc_crnn * 100, acc_trans * 100]
adv_accs_val = [adv_cnn * 100, adv_crnn * 100, adv_trans * 100]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, val_accs, width, label='Clean Validation', color='#2ecc71', edgecolor='black')
bars2 = ax.bar(x + width/2, adv_accs_val, width, label='Adversarial Test', color='#e74c3c', edgecolor='black')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Accuracy (%)')
ax.set_title('Comparacion de Robustez: Clean vs Adversarial')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
if adv_loader is not None:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, (name, _) in zip(axes, models_info):
        cm = adv_results[name]['confusion_matrix']
        if cm is not None:
            cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True).clip(min=1)
            im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
            ax.set_title(f'{name} - Adv Test')
            ax.set_xlabel('Predicho')
            ax.set_ylabel('Real')
            fig.colorbar(im, ax=ax, fraction=0.046)
    plt.suptitle('Matrices de Confusion Normalizadas - Test Adversarial', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Matrices de confusion no disponibles (sin labels adversariales)")

## Discusión

### ¿Qué modelo fue más robusto?

El **Transformer con RoPE** suele ser el más robusto frente a ataques adversariales debido a su capacidad de atender a patrones globales en el espectrograma, lo que lo hace menos sensible a perturbaciones localizadas. La CRNN ocupa un punto intermedio: las capas convolucionales extraen características locales robustas, y la BiLSTM modela dependencias temporales, pero puede verse afectada si el ataque distorsiona sistemáticamente ciertos marcos temporales. La CNN Baseline, al depender exclusivamente de filtros convolucionales locales sin modelado secuencial explícito, tiende a ser la más vulnerable.

### ¿Por qué una arquitectura maneja mejor las perturbaciones adversariales?

1. **Mecanismo de atención global**: El Transformer con RoPE puede establecer dependencias entre cualquier par de posiciones tiempo-frecuencia, lo que diluye el efecto de perturbaciones localizadas. Al rotar los embeddings posicionales en lugar de sumarlos, RoPE preserva la estructura geométrica del espacio temporal.
2. **Modelado secuencial en CRNN**: La BiLSTM captura la evolución temporal de los fonemas, lo que permite al modelo "corregir" interpretaciones erróneas en un marco temporal usando contexto adyacente.
3. **Sobrerrepresentación de patrones locales en CNN**: La CNN aprende parches locales altamente específicos. Un ataque adversarial que modifica un parche puede engañar fácilmente al clasificador.

### Limitaciones y trabajo futuro

- **Limitaciones**:
  - La evaluación adversarial se realizó sobre un conjunto de test pre-generado; no se generaron ataques ad-hoc (FGSM, PGD) para cada modelo.
  - Los modelos CRNN y Transformer se inicializaron con pesos aleatorios si no se encontraron los archivos `.pth`, por lo que las métricas reportadas pueden no reflejar el rendimiento real entrenado.
  - No se aplicaron técnicas de defensa adversarial (entrenamiento adversarial, suavizado aleatorio, etc.).

- **Trabajo futuro**:
  - Entrenar los modelos CRNN y Transformer desde cero con los mismos datos de entrenamiento.
  - Aplicar ataques adversariales de forma controlada (FGSM, PGD, Carlini-Wagner) para evaluar la robustez de forma más granular.
  - Implementar defensas como entrenamiento adversarial, destilación defensiva, o preprocesamiento de señal.
  - Explorar arquitecturas hybridas (CNN + Transformer) que combinen las ventajas de ambos paradigmas.